In [82]:
import json
import matplotlib.pyplot as plt
import numpy as np
import os

In [83]:
# Plot
plot_directory = 'result/plot_set'

In [84]:
colors = [
	'#FF0000',
	'#00FFFF',
	'#0000FF',
	'#00008B',
	'#ADD8E6',
	'#800080',
	'#7FFFD4',
	'#008000',
	'#FF00FF',
	'#FFC0CB',
	'#C0C0C0',
	'#FFA500',
	'#000000',
	'#800000',
]

In [85]:
def load_json_file(file_path):
	try:
		with open(file_path, 'r') as file:
			data = json.load(file)
		return data
	
	except Exception as e:
		print(f"An error occurred while loading the JSON file: {e}")
		return None

In [86]:
def extract_fpss(metric_list):
	return list(metric_list[list(metric_list.keys())[0]][0]['metric'].keys())

In [87]:
def to_accuracy_vector(accuracy_result_seq, fpss, type='F1'):
	accuracy_vector = []
	for fps in fpss:
		accuracy_vector.append(accuracy_result_seq[fps][type])
	
	return accuracy_vector

In [88]:
def plot_scatter(xs, ys, x_label, y_label, title, fig_size=(8, 8)):
	plt.figure(figsize=fig_size)
	plt.scatter(xs, ys, c=colors[0], s=30, alpha=0.5)
	plt.title(title)
	plt.xlabel(x_label)
	plt.ylabel(y_label)
	plt.show()

In [89]:
def plot_scatter_label(xs, ys, labels, x_label, y_label, title, fig_size=(8, 8)):
	plt.figure(figsize=fig_size)
	for i in range(len(xs)):
		plt.scatter(xs[i], ys[i], c=colors[labels[i]], s=30, alpha=0.5)
	plt.title(title)
	plt.xlabel(x_label)
	plt.ylabel(y_label)
	plt.show()

In [90]:
def filter_list_index(input_list, indices_to_remove):
	return [item for i, item in enumerate(input_list) if i not in indices_to_remove]

In [91]:
def round_float_to_sigfigs(number, sigfigs):
	return round(number, sigfigs)

## Plot

In [92]:
omv_features = ["Left-Top", "Right-Top", "Left-Bottom", "Right-Bottom", "Object-Amount", "Confidence", "IOU"]

In [93]:
plot_filenames = sorted(os.listdir(plot_directory))
plot_video_names = sorted(list(set([f.split('_')[0] for f in plot_filenames])))

In [94]:
fpss = extract_fpss(load_json_file(os.path.join(plot_directory, plot_video_names[0] + "_Accuracy_Result.json")))

In [95]:
omv_videos = []
acc_videos = []

for v in plot_video_names:
	omv_dict = {}
	for fps in fpss:
		omv_dict[fps] = []
	acc_list = []

	accuracy_result = load_json_file(os.path.join(plot_directory, v + "_Accuracy_Result.json"))
	movement_result = load_json_file(os.path.join(plot_directory, v + "_Movement_Result.json"))

	for class_idx in list(accuracy_result.keys()):
		for i in range(len(accuracy_result[class_idx])):
			accuracy_vector = to_accuracy_vector(accuracy_result[class_idx][i]['metric'], fpss)
			acc_list.append(accuracy_vector)

			for fps in fpss:
				movement_vector = movement_result[class_idx][i]['movement'][fps]
				omv_dict[fps].append(movement_vector)
	
	omv_videos.append(omv_dict)
	acc_videos.append(acc_list)

In [96]:
# Corr

# for i in range(len(plot_video_names)):
# 	video_name = plot_video_names[i]
# 	for k in range(len(fpss)):
# 		for j in range(len(omv_videos[0][fpss[0]][0])):	
# 			fps = fpss[k]

# 			omv_fps = list(np.array(omv_videos[i][fps])[:, j])
# 			acc_fps = list(np.array(acc_videos[i])[:, k])

# 			# Remove Outliers
# 			outlier_index = [l for l in range(len(omv_fps)) if omv_fps[l] == -1]
# 			omv_fps_clean = filter_list_index(omv_fps, outlier_index)
# 			acc_fps_clean = filter_list_index(acc_fps, outlier_index)

# 			title = f'{video_name}; OMV Feature {j} ({omv_features[j]}); FPS: {fps}'

# 			correlation_matrix = np.corrcoef(omv_fps_clean, acc_fps_clean)
# 			correlation_coefficient = correlation_matrix[0, 1]
# 			print(f"{title} -> Corr: {round_float_to_sigfigs(correlation_coefficient, 3)}")
# 			# plot_scatter(omv_fps_clean, acc_fps_clean, 'OMV Feature', 'ACC', title)

# 		print("")

# New Code

In [97]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import PolynomialFeatures

In [98]:
def transpose_array(arr):
    # Transpose the input array using zip and map it back to a list
    return [list(row) for row in zip(*arr)]

In [99]:
def my_train_test_split(x, y, test_size):
	split_point = int(len(x) * (1-test_size))
	return x[0:split_point], x[split_point:len(x)], y[0:split_point], y[split_point:len(x)]

In [100]:
def root_mean_squared_error(y_train, y_train_pred):
	return np.sqrt(mean_squared_error(y_train, y_train_pred))

In [101]:
OMV_FEATURE_INDEX = [5, 6]
TEST_SIZE = 0.2

In [102]:
# Dataset Build

dataset = {}

for i in range(len(plot_video_names)):
	video_name = plot_video_names[i]
	dataset[video_name] = {}
	
	for k in range(len(fpss)):
		fps = fpss[k]
		dataset[video_name][fps] = {}
		y_all = None
		x_all_t = []

		for j in range(len(omv_videos[0][fpss[0]][0])):	
			fps = fpss[k]

			omv_fps = list(np.array(omv_videos[i][fps])[:, j])
			acc_fps = list(np.array(acc_videos[i])[:, k])

			# Remove Outliers
			outlier_index = [l for l in range(len(omv_fps)) if omv_fps[l] == -1]
			omv_fps_clean = filter_list_index(omv_fps, outlier_index)
			acc_fps_clean = filter_list_index(acc_fps, outlier_index)

			if j == 0:
				y_all = acc_fps_clean.copy()
			if j in OMV_FEATURE_INDEX:
				x_all_t.append(omv_fps_clean.copy())
		
		x_all = transpose_array(x_all_t)
		dataset[video_name][fps]['y_all'] = y_all
		dataset[video_name][fps]['x_all'] = x_all
			

In [103]:
def lr_inference(x_train, x_test, y_train, y_test):
	lr_model = LinearRegression()
	lr_model.fit(x_train, y_train)

	y_train_pred = lr_model.predict(x_train)
	y_test_pred = lr_model.predict(x_test)

	train_mse = root_mean_squared_error(y_train, y_train_pred)
	test_mse = root_mean_squared_error(y_test, y_test_pred)
	
	# print(f"Linear Regression Train MSE: {round(train_mse, 4)}")
	# print(f"Linear Regression Test MSE: {round(test_mse, 4)}")

	return train_mse, test_mse

In [104]:
def pr_inference(x_train, x_test, y_train, y_test):
	poly_train = PolynomialFeatures(degree=3)
	poly_test = PolynomialFeatures(degree=3)
	x_train_poly = poly_train.fit_transform(x_train)
	x_test_poly = poly_test.fit_transform(x_test)

	pr_model = LinearRegression()
	pr_model.fit(x_train_poly, y_train)

	y_train_pred = pr_model.predict(x_train_poly)
	y_test_pred = pr_model.predict(x_test_poly)

	train_mse = root_mean_squared_error(y_train, y_train_pred)
	test_mse = root_mean_squared_error(y_test, y_test_pred)

	# print(f"Polynomial Regression Train MSE: {round(train_mse, 4)}")
	# print(f"Polynomial Regression Test MSE: {round(test_mse, 4)}")

	return train_mse, test_mse

In [105]:
def rfr_inference(x_train, x_test, y_train, y_test):
	rfr_model = RandomForestRegressor(n_estimators=100, random_state=42)
	rfr_model.fit(x_train, y_train)

	y_train_pred = rfr_model.predict(x_train)
	y_test_pred = rfr_model.predict(x_test)

	train_mse = root_mean_squared_error(y_train, y_train_pred)
	test_mse = root_mean_squared_error(y_test, y_test_pred)

	# print(f"Random Forest Regression Train MSE: {round(train_mse, 4)}")
	# print(f"Random Forest Regression Test MSE: {round(test_mse, 4)}")

	return train_mse, test_mse

In [106]:
def svr_inference(x_train, x_test, y_train, y_test):
	svr_model = SVR(kernel='rbf', C=1e3, gamma=0.1)
	svr_model.fit(x_train, y_train)

	y_train_pred = svr_model.predict(x_train)
	y_test_pred = svr_model.predict(x_test)

	train_mse = root_mean_squared_error(y_train, y_train_pred)
	test_mse = root_mean_squared_error(y_test, y_test_pred)

	# print(f"Support Vector Regression Train MSE: {round(train_mse, 4)}")
	# print(f"Support Vector Regression Test MSE: {round(test_mse, 4)}")

	return train_mse, test_mse

In [107]:
def knn_inference(x_train, x_test, y_train, y_test):
	knn_model = KNeighborsRegressor(n_neighbors=5)  # Using k = 5 neighbors
	knn_model.fit(x_train, y_train)

	y_train_pred = knn_model.predict(x_train)
	y_test_pred = knn_model.predict(x_test)

	train_mse = root_mean_squared_error(y_train, y_train_pred)
	test_mse = root_mean_squared_error(y_test, y_test_pred)

	# print(f"K-Nearest Neighbors Train MSE: {round(train_mse, 4)}")
	# print(f"K-Nearest Neighbors Test MSE: {round(test_mse, 4)}")

	return train_mse, test_mse

In [108]:
for k in range(len(fpss)):
	fps = fpss[k]

	print(f"FPS: {fps}")

	lr_train_mse_list = []
	lr_test_mse_list = []
	pr_train_mse_list = []
	pr_test_mse_list = []
	rfr_train_mse_list = []
	rfr_test_mse_list = []
	svr_train_mse_list = []
	svr_test_mse_list = []
	knn_train_mse_list = []
	knn_test_mse_list = []
	for i in range(len(plot_video_names)):
		video_name = plot_video_names[i]

		x_train = []
		x_test = []
		y_train = []
		y_test = []
		for l in range(len(plot_video_names)):
			if l == i:
				x_test.extend(dataset[plot_video_names[l]][fps]['x_all'])
				y_test.extend(dataset[plot_video_names[l]][fps]['y_all'])
			else:
				x_train.extend(dataset[plot_video_names[l]][fps]['x_all'])
				y_train.extend(dataset[plot_video_names[l]][fps]['y_all'])

		# LR
		lr_train_mse, lr_test_mse = lr_inference(x_train, x_test, y_train, y_test)
		lr_train_mse_list.append(lr_train_mse)
		lr_test_mse_list.append(lr_test_mse)

		# PR
		pr_train_mse, pr_test_mse = pr_inference(x_train, x_test, y_train, y_test)
		pr_train_mse_list.append(pr_train_mse)
		pr_test_mse_list.append(pr_test_mse)

		# RFR
		rfr_train_mse, rfr_test_mse = rfr_inference(x_train, x_test, y_train, y_test)
		rfr_train_mse_list.append(rfr_train_mse)
		rfr_test_mse_list.append(rfr_test_mse)

		# SVR
		svr_train_mse, svr_test_mse = svr_inference(x_train, x_test, y_train, y_test)
		svr_train_mse_list.append(svr_train_mse)
		svr_test_mse_list.append(svr_test_mse)

		# KNN
		knn_train_mse, knn_test_mse = knn_inference(x_train, x_test, y_train, y_test)
		knn_train_mse_list.append(knn_train_mse)
		knn_test_mse_list.append(knn_test_mse)
	
	print(f"LR")
	print(f"Average Train Accuracy: {round(np.average(np.array(lr_train_mse_list)), 4)}")
	print(f"Average Test Accuracy: {round(np.average(np.array(lr_test_mse_list)), 4)}")
	# print(np.array(lr_train_mse_list))
	# print(np.array(lr_test_mse_list))

	print(f"PR")
	print(f"Average Train Accuracy: {round(np.average(np.array(pr_train_mse_list)), 4)}")
	print(f"Average Test Accuracy: {round(np.average(np.array(pr_test_mse_list)), 4)}")
	# print(np.array(pr_train_mse_list))
	# print(np.array(pr_test_mse_list))

	print(f"RFR")
	print(f"Average Train Accuracy: {round(np.average(np.array(rfr_train_mse_list)), 4)}")
	print(f"Average Test Accuracy: {round(np.average(np.array(rfr_test_mse_list)), 4)}")
	# print(np.array(rfr_train_mse_list))
	# print(np.array(rfr_test_mse_list))

	print(f"SVR")
	print(f"Average Train Accuracy: {round(np.average(np.array(svr_train_mse_list)), 4)}")
	print(f"Average Test Accuracy: {round(np.average(np.array(svr_test_mse_list)), 4)}")
	# print(np.array(svr_train_mse_list))
	# print(np.array(svr_test_mse_list))
	
	print(f"KNN")
	print(f"Average Train Accuracy: {round(np.average(np.array(knn_train_mse_list)), 4)}")
	print(f"Average Test Accuracy: {round(np.average(np.array(knn_test_mse_list)), 4)}")
	# print(np.array(knn_train_mse_list))
	# print(np.array(knn_test_mse_list))

	print(f"")

FPS: 1
LR
Average Train Accuracy: 0.1604
Average Test Accuracy: 0.1683
PR
Average Train Accuracy: 0.1586
Average Test Accuracy: 0.1718
RFR
Average Train Accuracy: 0.0641
Average Test Accuracy: 0.1906
SVR
Average Train Accuracy: 0.1598
Average Test Accuracy: 0.1706
KNN
Average Train Accuracy: 0.1374
Average Test Accuracy: 0.1871

FPS: 2
LR
Average Train Accuracy: 0.1374
Average Test Accuracy: 0.1449
PR
Average Train Accuracy: 0.1342
Average Test Accuracy: 0.1459
RFR
Average Train Accuracy: 0.0526
Average Test Accuracy: 0.149
SVR
Average Train Accuracy: 0.1361
Average Test Accuracy: 0.1465
KNN
Average Train Accuracy: 0.1151
Average Test Accuracy: 0.1507

FPS: 3
LR
Average Train Accuracy: 0.1148
Average Test Accuracy: 0.1189
PR
Average Train Accuracy: 0.1111
Average Test Accuracy: 0.1186
RFR
Average Train Accuracy: 0.0452
Average Test Accuracy: 0.13
SVR
Average Train Accuracy: 0.114
Average Test Accuracy: 0.1222
KNN
Average Train Accuracy: 0.0985
Average Test Accuracy: 0.127

FPS: 5
LR
Av